# Import Packages

In [ ]:
%config Completer.use_jedi = False
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image
import seaborn as sns
import cv2
import random
import os
import imageio
import plotly.graph_objects as go
import plotly.express as px
import plotly.figure_factory as ff
from plotly.subplots import make_subplots
from collections import Counter

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import LocalOutlierFactor
from tensorflow.keras.callbacks import Callback, EarlyStopping,ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import accuracy_score, recall_score, precision_score, classification_report, confusion_matrix, plot_confusion_matrix,f1_score
from sklearn.model_selection import RandomizedSearchCV, cross_val_score, RepeatedStratifiedKFold
from imblearn.over_sampling import SMOTE

import tensorflow as tf
import tensorflow_addons as tfa
import keras
from keras.models import Sequential
from keras.layers import Dense, Dropout, Activation, Flatten
from keras.layers import Conv2D, MaxPooling2D, GlobalAveragePooling2D, BatchNormalization
from keras.applications import resnet
from tensorflow.keras.applications import EfficientNetB0, EfficientNetB1, EfficientNetB2, EfficientNetB3, EfficientNetB4, EfficientNetB5, EfficientNetB6, EfficientNetB7
from keras.applications.resnet import ResNet50
from keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array, array_to_img

In [ ]:
directory = r'/kaggle/input/iqothnccd-lung-cancer-dataset/The IQ-OTHNCCD lung cancer dataset/The IQ-OTHNCCD lung cancer dataset'

categories = ['Bengin cases', 'Malignant cases', 'Normal cases']

# Image Size Variations

In [ ]:
size_data = {}
for i in categories:
    path = os.path.join(directory, i)
    class_num = categories.index(i)
    temp_dict = {}
    for file in os.listdir(path):
        filepath = os.path.join(path, file)
        height, width, channels = imageio.imread(filepath).shape
        if str(height) + ' x ' + str(width) in temp_dict:
            temp_dict[str(height) + ' x ' + str(width)] += 1 
        else:
            temp_dict[str(height) + ' x ' + str(width)] = 1
    
    size_data[i] = temp_dict
        
size_data

In [ ]:
for i in categories:
    path = os.path.join(directory, i)
    class_num = categories.index(i)
    for file in os.listdir(path):
        filepath = os.path.join(path, file)
        print(i)
        img = cv2.imread(filepath, 0)
        plt.imshow(img)
        plt.show()
        break

# Image Preprocessing and Testing

In [ ]:
img_size = 256
for i in categories:
    cnt, samples = 0, 3
    fig, ax = plt.subplots(samples, 3, figsize=(15, 15))
    fig.suptitle(i)
    
    path = os.path.join(directory, i)
    class_num = categories.index(i)
    for curr_cnt, file in enumerate(os.listdir(path)):
        filepath = os.path.join(path, file)
        img = cv2.imread(filepath, 0)
        
        img0 = cv2.resize(img, (img_size, img_size))
        
        img1 = cv2.GaussianBlur(img0, (5, 5), 0)
        
        ax[cnt, 0].imshow(img)
        ax[cnt, 1].imshow(img0)
        ax[cnt, 2].imshow(img1)
        cnt += 1
        if cnt == samples:
            break
        
plt.show()

# Preparing Data

In [ ]:
import cv2
import numpy as np
from collections import Counter

data = []
img_size = 256

for i in categories:
    path = os.path.join(directory, i)
    class_num = categories.index(i)
    for file in os.listdir(path):
        filepath = os.path.join(path, file)
        img = cv2.imread(filepath)  # Load in RGB format by default
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # Convert from BGR to RGB
        img = cv2.resize(img, (img_size, img_size))
        data.append([img, class_num])

# Shuffle the data
random.shuffle(data)

# Prepare X and y
X, y = [], []
for feature, label in data:
    X.append(feature)
    y.append(label)

# Normalize X
X = np.array(X) / 255.0  # Normalize pixel values
y = np.array(y)

# Check the shapes and counts
print('X length:', len(X))
print('y counts:', Counter(y))

In [ ]:
X.shape

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, random_state=10, stratify=y)

print(len(X_train), X_train.shape)
print(len(X_valid), X_valid.shape)

# See the images

In [ ]:
import matplotlib.pyplot as plt

# Display images from X_train
plt.figure(figsize=(10, 10))
for i in range(10):  # Displaying 10 images
    plt.subplot(5, 5, i + 1)
    plt.imshow(X_train[i], cmap='gray')  # Assuming grayscale images
    plt.axis('off')
    plt.title(f"Class: {y_train[i]}")
plt.tight_layout()
plt.show()

# Display images from X_valid
plt.figure(figsize=(10, 10))
for i in range(10):  # Displaying 10 images
    plt.subplot(5, 5, i + 1)
    plt.imshow(X_valid[i], cmap='gray')  # Assuming grayscale images
    plt.axis('off')
    plt.title(f"Class: {y_valid[i]}")
plt.tight_layout()
plt.show()

# Applying SMOTE to oversample the data

In [ ]:
print(Counter(y_train), Counter(y_valid))

In [ ]:
print(len(X_train), X_train.shape)

X_train = X_train.reshape(X_train.shape[0], img_size*img_size*3)

print(len(X_train), X_train.shape)

In [ ]:
print('Before SMOTE:', Counter(y_train))
smote = SMOTE()
X_train_sampled, y_train_sampled = smote.fit_resample(X_train, y_train)
print('After SMOTE:', Counter(y_train_sampled))

In [ ]:
X_train = X_train.reshape(X_train.shape[0], img_size, img_size, 3)
X_train_sampled = X_train_sampled.reshape(X_train_sampled.shape[0], img_size, img_size, 3)

print(len(X_train), X_train.shape)
print(len(X_train_sampled), X_train_sampled.shape)

In [ ]:
# Display images from X_train_sampled
plt.figure(figsize=(10, 10))
for i in range(10):  # Displaying 10 images
    plt.subplot(5, 5, i + 1)
    plt.imshow(X_train_sampled[i], cmap='gray')  # Assuming grayscale images
    plt.axis('off')
    plt.title(f"Class: {y_train_sampled[i]}")
plt.tight_layout()
plt.show()

# Model Building with SMOTE data

In [ ]:
# Create checkpoint callback
checkpoint_path = "lung_cancer_classification_model_checkpoint"
checkpoint_callback = ModelCheckpoint(checkpoint_path,
                                      save_weights_only=True,
                                      monitor="val_accuracy",
                                      save_best_only=True)

# Setup EarlyStopping callback to stop training if model's val_loss doesn't improve for 3 epochs
early_stopping = EarlyStopping(monitor = "val_loss", # watch the val loss metric
                               patience = 5,
                               restore_best_weights = True) # if val loss decreases for 3 epochs in a row, stop training

reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6)

In [ ]:
from tensorflow.keras.applications import EfficientNetB7
from tensorflow.keras import layers, models, optimizers

# Instantiate the EfficientNetB7 model
base_model = EfficientNetB7(weights='imagenet', 
                            include_top=False, 
                            input_shape=(256,256,3),
                            classes= 3)

# Freeze the base model layers
base_model.trainable = False

    
# Create a new model on top of EfficientNetB0
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.BatchNormalization(),
    layers.Dense(3, activation='softmax') 
])

model.summary()

In [ ]:
# Compile the model
model.compile(
    optimizer=optimizers.Adam(lr=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# Train the model
history = model.fit(
    X_train_sampled, 
    y_train_sampled,
    #epochs= 25,  
    epochs= 25,  
    validation_data=(X_valid, y_valid),
    callbacks=[
        early_stopping,
        checkpoint_callback,
        reduce_lr
    ]
)

# Results

In [ ]:
y_pred = model.predict(X_valid, verbose=1)
y_pred_bool = np.argmax(y_pred, axis=1)

print(classification_report(y_valid, y_pred_bool))

print(confusion_matrix(y_true=y_valid, y_pred=y_pred_bool))

In [ ]:
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Validation')
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend()
plt.show()

plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Validation')
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend()
plt.show()

# Data Augmentation

In [ ]:
BATCH_SIZE = 8

# Add Image augmentation to our generator
train_datagen = ImageDataGenerator(rescale=1./255,
                                   rotation_range=360,
                                   horizontal_flip=True,
                                   vertical_flip=True,
                                   width_shift_range=0.1,
                                   height_shift_range=0.1,
                                   zoom_range=(0.75,1),
                                   brightness_range=(0.75,1.25)
                                  )

val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow(X_train, y_train, batch_size=BATCH_SIZE)
val_generator = train_datagen.flow(X_valid, y_valid, batch_size=BATCH_SIZE, shuffle= False)

# Fine Tuned Model

In [ ]:
from tensorflow.keras.applications import EfficientNetB7
from tensorflow.keras import layers, models, optimizers


# Instantiate the EfficientNetB7 model
base_model = EfficientNetB7(weights='imagenet', 
                            include_top=False, 
                            input_shape=(256,256,3),
                            classes= 3)

# Unfreeze some layers for fine-tuning
base_model.trainable = True

# Fine-tune from this layer onwards
fine_tune_at = 100

# Freeze all layers before the fine-tune starting layer
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False
    
# Create a new model on top of EfficientNetB0
model3 = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(3, activation='softmax') 
])

model3.summary()

In [ ]:
# Compile the model
model3.compile(
    optimizer=optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# Train the model
history = model3.fit(
    train_generator, 
    epochs= 25,  
    validation_data=val_generator,
    callbacks=[
        early_stopping,
        checkpoint_callback,
        reduce_lr
    ]
)

In [ ]:
y_pred = model3.predict(X_valid, verbose=1)
y_pred_bool = np.argmax(y_pred, axis=1)


report = classification_report(y_valid, y_pred_bool)
conf_mat = confusion_matrix(y_true=y_valid, y_pred=y_pred_bool)

In [ ]:
print("\nClassification Report:")
print(conf_mat)
print("\nClassification Report:")
print(report)

In [ ]:

# 1. ROC Curve and AUC calculation
from sklearn.metrics import roc_curve, auc, roc_auc_score
from scipy import interp
from itertools import cycle

# Calculate ROC curves for multi-class problem
y_pred_proba = model3.predict(X_valid)
n_classes = 3

# Calculate ROC curve and ROC area for each class
fpr = dict()
tpr = dict()
roc_auc = dict()

for i in range(n_classes):
    # Convert validation labels to one-hot encoding
    y_valid_bin = np.zeros((len(y_valid), n_classes))
    for j in range(len(y_valid)):
        y_valid_bin[j, y_valid[j]] = 1
    
    fpr[i], tpr[i], _ = roc_curve(y_valid_bin[:, i], y_pred_proba[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Calculate micro-average ROC curve and ROC area
fpr["micro"], tpr["micro"], _ = roc_curve(y_valid_bin.ravel(), y_pred_proba.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

# Plot all ROC curves
plt.figure(figsize=(12, 8))
plt.plot(fpr["micro"], tpr["micro"],
         label=f'Micro-average ROC curve (AUC = {roc_auc["micro"]:.2f})',
         color='deeppink', linestyle=':', linewidth=4)

colors = cycle(['blue', 'red', 'green'])
for i, color in zip(range(n_classes), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=2,
             label=f'ROC curve {categories[i]} (AUC = {roc_auc[i]:.2f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Multi-class ROC Curves')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

# 2. Enhanced confusion matrix visualization
plt.figure(figsize=(10, 8))
conf_mat_norm = conf_mat.astype('float') / conf_mat.sum(axis=1)[:, np.newaxis]

# Plot original confusion matrix
plt.figure(figsize=(12, 8))
sns.heatmap(conf_mat, annot=True, fmt='d', cmap='Blues', 
            xticklabels=categories,  
            yticklabels=categories)
plt.xlabel('Predicted Class')
plt.ylabel('True Class')
plt.title('Confusion Matrix (Raw Counts)')
plt.savefig("Confusion Matrix (Raw Counts)")
plt.show()

# Plot normalized confusion matrix
plt.figure(figsize=(12, 8))
sns.heatmap(conf_mat_norm, annot=True, fmt='.2f', cmap='Blues', 
            xticklabels=categories,  
            yticklabels=categories)
plt.xlabel('Predicted Class')
plt.ylabel('True Class')
plt.title('Confusion Matrix (Normalized)')

plt.tight_layout()
plt.savefig("Confusion Matrix (Normalized)")
plt.show()





In [ ]:
# ... existing code ...

# 3. Bar chart for precision, recall, F1-score
from sklearn.metrics import precision_recall_fscore_support

class_precision, class_recall, class_f1, _ = precision_recall_fscore_support(y_valid, y_pred_bool, average=None)

# Create bar chart of per-class metrics
metrics_df = pd.DataFrame({
    'Precision': class_precision,
    'Recall': class_recall,
    'F1-Score': class_f1
}, index=categories)

plt.figure(figsize=(12, 6))
metrics_df.plot(kind='bar', ylim=[0, 1.1], rot=0)
plt.title('Per-Class Performance Metrics')
plt.ylabel('Score')
plt.xlabel('Class')
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

# 4. Class imbalance visualization
plt.figure(figsize=(10, 5))
class_counts = np.bincount(y_valid)
plt.bar(range(len(categories)), class_counts, color=['skyblue', 'lightgreen', 'salmon'])
plt.xticks(range(len(categories)), categories)
plt.title('Number of Samples per Class in Validation Set')
plt.xlabel('Class')
plt.ylabel('Number of Samples')
plt.grid(True, linestyle='--', alpha=0.6, axis='y')
plt.tight_layout()
plt.show()

# 5. Precision-Recall curves
from sklearn.metrics import precision_recall_curve, average_precision_score

# Use different variable names to avoid conflict
pr_precision = dict()
pr_recall = dict()
average_precision = dict()

for i in range(n_classes):
    y_valid_bin = np.zeros((len(y_valid), n_classes))
    for j in range(len(y_valid)):
        y_valid_bin[j, y_valid[j]] = 1
        
    pr_precision[i], pr_recall[i], _ = precision_recall_curve(y_valid_bin[:, i], y_pred_proba[:, i])
    average_precision[i] = average_precision_score(y_valid_bin[:, i], y_pred_proba[:, i])

# Calculate micro-average precision-recall curve
pr_precision["micro"], pr_recall["micro"], _ = precision_recall_curve(y_valid_bin.ravel(), y_pred_proba.ravel())
average_precision["micro"] = average_precision_score(y_valid_bin.ravel(), y_pred_proba.ravel())

# Plot precision-recall curves
plt.figure(figsize=(12, 8))
plt.plot(pr_recall["micro"], pr_precision["micro"],
         label=f'Micro-average PR curve (AP = {average_precision["micro"]:.2f})',
         color='gold', linestyle=':', linewidth=4)

colors = cycle(['blue', 'red', 'green'])
for i, color in zip(range(n_classes), colors):
    plt.plot(pr_recall[i], pr_precision[i], color=color, lw=2,
             label=f'PR curve {categories[i]} (AP = {average_precision[i]:.2f})')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Multi-class Precision-Recall Curves')
plt.legend(loc="lower left")
plt.grid(True)
plt.show()

# 6. Generate model performance summary report
print("\n=== Model Performance Summary ===")
print(f"Micro-average AUC: {roc_auc['micro']:.4f}")
print(f"Class average AUC: {np.mean([roc_auc[i] for i in range(n_classes)]):.4f}")
print(f"Micro-average Precision-Recall Area: {average_precision['micro']:.4f}")

# Print detailed metrics for each class
for i in range(n_classes):
    print(f"\nClass: {categories[i]}")
    print(f"AUC: {roc_auc[i]:.4f}")
    print(f"Precision: {class_precision[i]:.4f}")
    print(f"Recall: {class_recall[i]:.4f}")
    print(f"F1-Score: {class_f1[i]:.4f}")


In [ ]:
# 添加一个更通用的Grad-CAM实现
import tensorflow as tf
import matplotlib.cm as cm
import matplotlib.pyplot as plt

def get_img_array(img, size):
    # 扩展维度并确保使用正确的格式
    img_array = np.expand_dims(img, axis=0)
    return img_array

def make_gradcam_heatmap(img_array, model, last_conv_layer_index=-4, pred_index=None):
    """
    创建一个更通用的Grad-CAM热力图实现
    
    参数:
        img_array: 预处理后的输入图像数组
        model: 训练好的模型
        last_conv_layer_index: 最后一个卷积层在base_model中的索引(负数表示从末尾开始)
        pred_index: 要生成热力图的类别索引(默认为预测的类别)
    
    返回:
        Grad-CAM热力图
    """
    # 首先预测类别
    preds = model.predict(img_array)
    if pred_index is None:
        pred_index = np.argmax(preds[0])
    
    # 获取模型中的base_model
    base_model = model.layers[0]
    
    # 获取最后一个卷积层
    last_conv_layer = base_model.layers[last_conv_layer_index]
    print(f"Using layer '{last_conv_layer.name}' for Grad-CAM")
    
    # 创建一个特征提取模型
    feature_model = tf.keras.models.Model(
        inputs=base_model.inputs,
        outputs=base_model.get_layer(last_conv_layer.name).output
    )
    
    # 创建一个分类模型
    classification_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=model.outputs
    )
    
    # 使用特征提取模型获取卷积层的输出
    with tf.GradientTape() as tape:
        # 计算最后一个卷积层的输出
        conv_output = feature_model(img_array)
        tape.watch(conv_output)
        
        # 使用这个输出创建一个带有可监视梯度的新模型
        # 这里我们使用一个简化模型来模拟从卷积层输出到分类结果的过程
        model_input = tf.keras.Input(shape=conv_output.shape[1:])
        x = model_input
        for layer in model.layers[1:]:  # 跳过base_model层
            x = layer(x)
        simplified_model = tf.keras.Model(inputs=model_input, outputs=x)
        
        # 计算预测结果
        pred_result = simplified_model(conv_output)
        class_output = pred_result[:, pred_index]
    
    # 获取类别相对于卷积层输出的梯度
    grads = tape.gradient(class_output, conv_output)
    
    # 计算梯度的全局平均值
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    
    # 将卷积层输出与我们计算的权重相乘
    conv_output = conv_output[0]
    heatmap = tf.reduce_sum(tf.multiply(pooled_grads, conv_output), axis=-1)
    
    # 标准化热力图
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    
    return heatmap.numpy()

def save_and_display_gradcam(img, heatmap, alpha=0.4, cam_path="cam.jpg"):
    """
    叠加热力图到原始图像上
    
    参数:
        img: 原始图像(RGB)
        heatmap: Grad-CAM热力图
        alpha: 热力图透明度系数
        cam_path: 保存结果图像的路径
    """
    # 调整热力图大小以匹配原始图像
    heatmap = np.uint8(255 * heatmap)
    
    # 将热力图应用到jet颜色图
    jet = cm.get_cmap("jet")
    jet_colors = jet(np.arange(256))[:, :3]
    jet_heatmap = jet_colors[heatmap]
    
    # 创建一个具有RGB彩色热力图的图像
    jet_heatmap = tf.keras.preprocessing.image.array_to_img(jet_heatmap)
    jet_heatmap = jet_heatmap.resize((img.shape[1], img.shape[0]))
    jet_heatmap = tf.keras.preprocessing.image.img_to_array(jet_heatmap)
    
    # 将热力图叠加到原始图像上
    superimposed_img = jet_heatmap * alpha + img
    superimposed_img = tf.keras.preprocessing.image.array_to_img(superimposed_img)
    
    return superimposed_img

# 测试一个样本的Grad-CAM
try:
    # 选择一个样本
    sample_idx = 20  # 可以选择任何有效的样本索引
    img = X_valid[sample_idx]
    img_array = np.expand_dims(img, axis=0)
    
    # 预测类别
    preds = model3.predict(img_array)
    predicted_class = np.argmax(preds[0])
    true_class = y_valid[sample_idx]
    
    print(f"Sample image - True class: {categories[true_class]}, Predicted class: {categories[predicted_class]}")
    
    # 尝试不同的last_conv_layer_index值
    for layer_index in [-1, -2, -3, -4, -5]:
        try:
            print(f"\nTrying layer index {layer_index}...")
            # 生成Grad-CAM热力图
            heatmap = make_gradcam_heatmap(img_array, model3, last_conv_layer_index=layer_index, pred_index=predicted_class)
            
            # 显示原始图像
            plt.figure(figsize=(16, 5))
            
            plt.subplot(1, 3, 1)
            plt.imshow(img)
            plt.title(f"Original Image\nTrue: {categories[true_class]}")
            plt.axis('off')
            
            # 显示热力图
            plt.subplot(1, 3, 2)
            plt.imshow(heatmap, cmap='jet')
            plt.title(f"Grad-CAM Heatmap\nPred: {categories[predicted_class]}")
            plt.axis('off')
            
            # 显示叠加图
            plt.subplot(1, 3, 3)
            superimposed = save_and_display_gradcam(img, heatmap)
            plt.imshow(superimposed)
            plt.title("Grad-CAM Overlay")
            plt.axis('off')
            
            plt.tight_layout()
            plt.show()
            
            # 找到一个有效的层后就退出
            print(f"Successfully generated Grad-CAM with layer index {layer_index}")
            break
            
        except Exception as e:
            print(f"Error with layer index {layer_index}: {str(e)}")
    
except Exception as e:
    print(f"Error in Grad-CAM generation: {str(e)}")

# 如果成功找到了适合的层索引，可以继续处理更多样本
# 这里我们假设layer_index成功生成了热力图
if 'heatmap' in locals():
    # 成功生成了热力图，可以继续处理更多样本
    successful_layer_index = layer_index
    
    # 为每个类别选择2个样本
    sample_indices = []
    for class_idx in range(3):
        class_indices = np.where(y_valid == class_idx)[0]
        if len(class_indices) >= 2:
            sample_indices.extend(class_indices[:2])
    
    # 创建一个图形来显示Grad-CAM可视化
    plt.figure(figsize=(15, 5 * len(sample_indices)))
    
    for i, idx in enumerate(sample_indices):
        try:
            # 获取图像
            img = X_valid[idx]
            img_array = np.expand_dims(img, axis=0)
            
            # 生成模型预测
            preds = model3.predict(img_array)
            predicted_class = np.argmax(preds[0])
            true_class = y_valid[idx]
            
            # 生成Grad-CAM热力图
            heatmap = make_gradcam_heatmap(img_array, model3, 
                                          last_conv_layer_index=successful_layer_index, 
                                          pred_index=predicted_class)
            
            # 显示图像
            plt.subplot(len(sample_indices), 3, i * 3 + 1)
            plt.imshow(img)
            plt.title(f"Original Image\nTrue: {categories[true_class]}")
            plt.axis('off')
            
            plt.subplot(len(sample_indices), 3, i * 3 + 2)
            plt.imshow(heatmap, cmap='jet')
            plt.title(f"Grad-CAM Heatmap\nPred: {categories[predicted_class]}")
            plt.axis('off')
            
            plt.subplot(len(sample_indices), 3, i * 3 + 3)
            superimposed = save_and_display_gradcam(img, heatmap)
            plt.imshow(superimposed)
            plt.title("Grad-CAM Overlay")
            plt.axis('off')
            
        except Exception as e:
            print(f"Error processing sample {idx}: {str(e)}")
            # 显示错误消息
            plt.subplot(len(sample_indices), 3, i * 3 + 1)
            plt.imshow(img)
            plt.title(f"Original Image\nTrue: {categories[true_class]}")
            plt.axis('off')
            
            plt.subplot(len(sample_indices), 3, i * 3 + 2)
            plt.text(0.5, 0.5, f"Error generating Grad-CAM", 
                     horizontalalignment='center', verticalalignment='center')
            plt.axis('off')
    
    plt.tight_layout()
    plt.show()


In [ ]:
# 设置全局图表风格
plt.style.use('seaborn-whitegrid')
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 10
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['figure.titlesize'] = 16
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.spines.top'] = False


In [ ]:
# 整合模型训练历史和性能评估
def plot_training_evaluation(history, y_valid, y_pred, y_pred_bool, categories):
    fig = plt.figure(figsize=(18, 12))
    
    # 1. 训练历史曲线
    ax1 = plt.subplot2grid((2, 3), (0, 0))
    ax1.plot(history.history['accuracy'], label='Train', color='#2077B4', linewidth=2)
    ax1.plot(history.history['val_accuracy'], label='Validation', color='#FF7F0E', linewidth=2)
    ax1.set_title('Model Accuracy')
    ax1.set_ylabel('Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.legend(frameon=True)
    ax1.grid(True, linestyle='--', alpha=0.6)
    
    ax2 = plt.subplot2grid((2, 3), (0, 1))
    ax2.plot(history.history['loss'], label='Train', color='#2077B4', linewidth=2)
    ax2.plot(history.history['val_loss'], label='Validation', color='#FF7F0E', linewidth=2)
    ax2.set_title('Model Loss')
    ax2.set_ylabel('Loss')
    ax2.set_xlabel('Epoch')
    ax2.legend(frameon=True)
    ax2.grid(True, linestyle='--', alpha=0.6)
    
    # 2. 混淆矩阵
    ax3 = plt.subplot2grid((2, 3), (0, 2))
    conf_mat = confusion_matrix(y_true=y_valid, y_pred=y_pred_bool)
    conf_mat_norm = conf_mat.astype('float') / conf_mat.sum(axis=1)[:, np.newaxis]
    sns.heatmap(conf_mat_norm, annot=True, fmt='.2f', cmap='Blues', 
                xticklabels=categories, yticklabels=categories, ax=ax3)
    ax3.set_xlabel('Predicted Class')
    ax3.set_ylabel('True Class')
    ax3.set_title('Normalized Confusion Matrix')
    
    # 3. 性能指标条形图
    ax4 = plt.subplot2grid((2, 3), (1, 0), colspan=2)
    class_precision, class_recall, class_f1, _ = precision_recall_fscore_support(y_valid, y_pred_bool, average=None)
    
    x = np.arange(len(categories))
    width = 0.25
    
    ax4.bar(x - width, class_precision, width, label='Precision', color='#2077B4', alpha=0.7)
    ax4.bar(x, class_recall, width, label='Recall', color='#FF7F0E', alpha=0.7)
    ax4.bar(x + width, class_f1, width, label='F1-Score', color='#2CA02C', alpha=0.7)
    
    ax4.set_ylim([0, 1.1])
    ax4.set_ylabel('Score')
    ax4.set_title('Performance Metrics by Class')
    ax4.set_xticks(x)
    ax4.set_xticklabels(categories)
    ax4.legend(frameon=True)
    ax4.grid(True, linestyle='--', alpha=0.6, axis='y')
    
    # 4. ROC曲线 (仅显示微平均)
    ax5 = plt.subplot2grid((2, 3), (1, 2))
    ax5.plot(fpr["micro"], tpr["micro"], 
             label=f'Micro-avg AUC = {roc_auc["micro"]:.2f}',
             color='deeppink', linewidth=2)
    ax5.plot([0, 1], [0, 1], 'k--', lw=1)
    ax5.set_xlim([0.0, 1.0])
    ax5.set_ylim([0.0, 1.05])
    ax5.set_xlabel('False Positive Rate')
    ax5.set_ylabel('True Positive Rate')
    ax5.set_title('ROC Curve')
    ax5.legend(loc="lower right")
    ax5.grid(True, linestyle='--', alpha=0.6)
    
    plt.tight_layout()
    plt.savefig("model_performance_summary.png", dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
# 改进Grad-CAM可视化展示
def plot_gradcam_comparison(model, X_valid, y_valid, categories, n_samples=2):
    """为每个类别生成对比性的Grad-CAM可视化"""
    fig = plt.figure(figsize=(15, n_samples * 7))
    
    # 为每个类别选择n_samples个样本
    sample_indices = []
    for class_idx in range(len(categories)):
        class_indices = np.where(y_valid == class_idx)[0]
        if len(class_indices) >= n_samples:
            sample_indices.extend(class_indices[:n_samples])
    
    for i, idx in enumerate(sample_indices):
        # 获取图像和预测
        img = X_valid[idx]
        img_array = np.expand_dims(img, axis=0)
        preds = model.predict(img_array)
        predicted_class = np.argmax(preds[0])
        true_class = y_valid[idx]
        
        # 生成预测类别的热力图
        try:
            heatmap = make_gradcam_heatmap(img_array, model, last_conv_layer_index=-4, pred_index=predicted_class)
            
            # 展示原始图像
            plt.subplot(len(sample_indices), 4, i * 4 + 1)
            plt.imshow(img)
            plt.title(f"Original\nTrue: {categories[true_class]}\nPred: {categories[predicted_class]}")
            plt.axis('off')
            
            # 展示热力图
            plt.subplot(len(sample_indices), 4, i * 4 + 2)
            plt.imshow(heatmap, cmap='jet')
            plt.title(f"Heatmap")
            plt.axis('off')
            
            # 展示叠加图
            plt.subplot(len(sample_indices), 4, i * 4 + 3)
            superimposed = save_and_display_gradcam(img, heatmap, alpha=0.5)
            plt.imshow(superimposed)
            plt.title(f"Overlay")
            plt.axis('off')
            
            # 添加预测概率条形图
            plt.subplot(len(sample_indices), 4, i * 4 + 4)
            plt.barh(categories, preds[0], color=['skyblue', 'lightgreen', 'salmon'])
            plt.xlim([0, 1])
            plt.title(f"Class Probabilities")
            
        except Exception as e:
            print(f"Error processing sample {idx}: {str(e)}")
    
    plt.tight_layout()
    plt.savefig("gradcam_visualization.png", dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
from sklearn.calibration import calibration_curve

def plot_calibration_curves(model, X_valid, y_valid, categories):
    """绘制模型校准曲线"""
    y_pred_proba = model.predict(X_valid)
    n_classes = len(categories)
    
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # 1. 可靠性图 (Reliability Diagram)
    ax1 = axes[0]
    ax1.plot([0, 1], [0, 1], "k:", label="Perfectly calibrated")
    
    for i in range(n_classes):
        # 将验证集标签转换为二元形式(当前类别为1，其他为0)
        y_valid_bin = (y_valid == i).astype(int)
        
        # 计算校准曲线
        prob_true, prob_pred = calibration_curve(y_valid_bin, y_pred_proba[:, i], n_bins=10)
        
        # 绘制校准曲线
        ax1.plot(prob_pred, prob_true, "s-", label=f"{categories[i]}")
    
    ax1.set_xlabel("Mean predicted probability")
    ax1.set_ylabel("Fraction of positives")
    ax1.set_title("Calibration curves")
    ax1.legend(loc="lower right")
    ax1.grid(True, linestyle='--', alpha=0.6)
    
    # 2. 预测概率直方图
    ax2 = axes[1]
    for i in range(n_classes):
        # 找出真正属于该类别的样本
        class_samples = y_pred_proba[y_valid == i, i]
        ax2.hist(class_samples, bins=20, alpha=0.5, label=f"{categories[i]}")
    
    ax2.set_xlabel("Predicted probability")
    ax2.set_ylabel("Count")
    ax2.set_title("Prediction probability distribution")
    ax2.legend(loc="upper center")
    ax2.grid(True, linestyle='--', alpha=0.6)
    
    plt.tight_layout()
    plt.savefig("calibration_curves.png", dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
def visualize_feature_importance(model, X_valid, y_valid, categories, n_samples=50):
    """通过多个样本的Grad-CAM热力图分析平均特征重要性"""
    # 为每个类别随机选择样本
    selected_indices = []
    for class_idx in range(len(categories)):
        class_indices = np.where(y_valid == class_idx)[0]
        if len(class_indices) > n_samples // len(categories):
            selected_indices.extend(np.random.choice(class_indices, n_samples // len(categories), replace=False))
        else:
            selected_indices.extend(class_indices)
    
    # 初始化热力图累积器
    heatmap_avg = np.zeros((img_size, img_size))
    success_count = 0
    
    # 生成并累积热力图
    for idx in selected_indices:
        try:
            img = X_valid[idx]
            img_array = np.expand_dims(img, axis=0)
            
            # 预测类别
            preds = model.predict(img_array)
            predicted_class = np.argmax(preds[0])
            
            # 生成热力图
            heatmap = make_gradcam_heatmap(img_array, model, last_conv_layer_index=-4, pred_index=predicted_class)
            heatmap_avg += heatmap
            success_count += 1
        except Exception as e:
            print(f"Error processing sample {idx}: {str(e)}")
    
    # 计算平均值并标准化
    if success_count > 0:
        heatmap_avg /= success_count
        heatmap_avg = heatmap_avg / np.max(heatmap_avg)
    
    # 创建可视化
    plt.figure(figsize=(12, 6))
    
    plt.subplot(1, 2, 1)
    plt.imshow(heatmap_avg, cmap='jet')
    plt.colorbar(label='Average Importance')
    plt.title(f"Average Feature Importance\nacross {success_count} samples")
    plt.axis('off')
    
    # 创建示例性的叠加图
    plt.subplot(1, 2, 2)
    # 使用一个示例图像
    example_idx = selected_indices[0]
    example_img = X_valid[example_idx]
    # 叠加平均热力图
    superimposed = save_and_display_gradcam(example_img, heatmap_avg, alpha=0.7)
    plt.imshow(superimposed)
    plt.title("Example with Average Importance Overlay")
    plt.axis('off')
    
    plt.tight_layout()
    plt.savefig("feature_importance.png", dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
def generate_comprehensive_report(model, X_train, y_train, X_valid, y_valid, categories, history):
    """生成综合性模型评估报告"""
    print("\n" + "="*50)
    print("COMPREHENSIVE MODEL EVALUATION REPORT")
    print("="*50)
    
    # 1. 数据集摘要
    print("\n--- DATASET SUMMARY ---")
    print(f"Training set: {X_train.shape[0]} samples")
    print(f"Validation set: {X_valid.shape[0]} samples")
    print("\nClass distribution:")
    for i, category in enumerate(categories):
        print(f"  • {category}: {np.sum(y_train == i)} (train), {np.sum(y_valid == i)} (validation)")
    
    # 2. 模型架构摘要
    print("\n--- MODEL ARCHITECTURE ---")
    model.summary()
    
    # 3. 训练过程摘要
    print("\n--- TRAINING SUMMARY ---")
    best_val_acc = max(history.history['val_accuracy'])
    best_epoch = np.argmax(history.history['val_accuracy']) + 1
    print(f"Best validation accuracy: {best_val_acc:.4f} (epoch {best_epoch})")
    print(f"Final training accuracy: {history.history['accuracy'][-1]:.4f}")
    print(f"Final validation accuracy: {history.history['val_accuracy'][-1]:.4f}")
    
    # 4. 性能指标
    print("\n--- PERFORMANCE METRICS ---")
    y_pred = model.predict(X_valid)
    y_pred_bool = np.argmax(y_pred, axis=1)
    
    # 计算整体指标
    accuracy = accuracy_score(y_valid, y_pred_bool)
    precision_micro = precision_score(y_valid, y_pred_bool, average='micro')
    recall_micro = recall_score(y_valid, y_pred_bool, average='micro')
    f1_micro = f1_score(y_valid, y_pred_bool, average='micro')
    
    print(f"Overall accuracy: {accuracy:.4f}")
    print(f"Micro-average precision: {precision_micro:.4f}")
    print(f"Micro-average recall: {recall_micro:.4f}")
    print(f"Micro-average F1-score: {f1_micro:.4f}")
    
    # 计算每个类别的指标
    class_precision, class_recall, class_f1, _ = precision_recall_fscore_support(y_valid, y_pred_bool, average=None)
    
    print("\nPer-class metrics:")
    for i, category in enumerate(categories):
        print(f"  • {category}:")
        print(f"      Precision: {class_precision[i]:.4f}")
        print(f"      Recall: {class_recall[i]:.4f}")
        print(f"      F1-score: {class_f1[i]:.4f}")
    
    # 5. 混淆矩阵
    conf_mat = confusion_matrix(y_true=y_valid, y_pred=y_pred_bool)
    print("\n--- CONFUSION MATRIX ---")
    print(conf_mat)
    
    # 6. ROC AUC 计算
    y_valid_bin = np.zeros((len(y_valid), len(categories)))
    for j in range(len(y_valid)):
        y_valid_bin[j, y_valid[j]] = 1
        
    # 计算每个类别的ROC曲线和ROC面积
    fpr = dict()
    tpr = dict()
    roc_auc = dict()
    
    for i in range(len(categories)):
        fpr[i], tpr[i], _ = roc_curve(y_valid_bin[:, i], y_pred[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
    
    # 计算微平均ROC曲线和ROC面积
    fpr["micro"], tpr["micro"], _ = roc_curve(y_valid_bin.ravel(), y_pred.ravel())
    roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])
    
    print("\n--- ROC AUC SCORES ---")
    print(f"Micro-average AUC: {roc_auc['micro']:.4f}")
    for i, category in enumerate(categories):
        print(f"  • {category}: {roc_auc[i]:.4f}")
    
    print("\n" + "="*50)


In [ ]:
# 在模型训练和初步评估之后调用
# 确保已经计算了y_pred, y_pred_bool和conf_mat

# 添加整合图表展示模型表现
plot_training_evaluation(history, y_valid, y_pred, y_pred_bool, categories)

# 保存详细性能指标到CSV文件
metrics_df = pd.DataFrame({
    'Class': categories,
    'Precision': class_precision,
    'Recall': class_recall,
    'F1-Score': class_f1,
    'Support': np.bincount(y_valid)[:len(categories)]
})
metrics_df.to_csv("model_performance_metrics.csv", index=False)
print("Saved performance metrics to model_performance_metrics.csv")


In [ ]:
# 在Grad-CAM实现之后，找到成功的layer_index后调用
if 'heatmap' in locals():
    successful_layer_index = layer_index
    
    # 生成每个类别的Grad-CAM可视化对比
    print("\nGenerating enhanced Grad-CAM visualizations...")
    plot_gradcam_comparison(model3, X_valid, y_valid, categories, n_samples=2)
    
    # 展示误分类样本的Grad-CAM
    misclassified = np.where(y_pred_bool != y_valid)[0]
    if len(misclassified) > 0:
        print(f"\nAnalyzing misclassified samples with Grad-CAM (found {len(misclassified)})")
        # 选择最多5个误分类样本
        misclassified_sample_indices = misclassified[:min(5, len(misclassified))]
        
        plt.figure(figsize=(16, 4*len(misclassified_sample_indices)))
        for i, idx in enumerate(misclassified_sample_indices):
            try:
                # 获取图像
                img = X_valid[idx]
                img_array = np.expand_dims(img, axis=0)
                
                # 预测
                preds = model3.predict(img_array)
                predicted_class = np.argmax(preds[0])
                true_class = y_valid[idx]
                
                # 生成预测类别和真实类别的热力图
                heatmap_pred = make_gradcam_heatmap(img_array, model3, last_conv_layer_index=successful_layer_index, pred_index=predicted_class)
                heatmap_true = make_gradcam_heatmap(img_array, model3, last_conv_layer_index=successful_layer_index, pred_index=true_class)
                
                # 显示原始图像
                plt.subplot(len(misclassified_sample_indices), 4, i*4+1)
                plt.imshow(img)
                plt.title(f"Original Image\nTrue: {categories[true_class]}\nPred: {categories[predicted_class]}")
                plt.axis('off')
                
                # 显示真实类别热力图
                plt.subplot(len(misclassified_sample_indices), 4, i*4+2)
                plt.imshow(heatmap_true, cmap='jet')
                plt.title(f"True Class Heatmap\n{categories[true_class]}")
                plt.axis('off')
                
                # 显示预测类别热力图
                plt.subplot(len(misclassified_sample_indices), 4, i*4+3)
                plt.imshow(heatmap_pred, cmap='jet')
                plt.title(f"Predicted Class Heatmap\n{categories[predicted_class]}")
                plt.axis('off')
                
                # 显示预测概率
                plt.subplot(len(misclassified_sample_indices), 4, i*4+4)
                plt.barh(categories, preds[0])
                plt.xlim([0, 1])
                plt.title("Class Probabilities")
                
            except Exception as e:
                print(f"Error analyzing misclassified sample {idx}: {str(e)}")
        
        plt.tight_layout()
        plt.savefig("misclassified_gradcam_analysis.png", dpi=300, bbox_inches='tight')
        plt.show()


In [ ]:
# 生成综合性评估报告
print("\nGenerating comprehensive evaluation report...")
generate_comprehensive_report(model3, X_train, y_train, X_valid, y_valid, categories, history)

# 将报告导出为文本文件
import sys
from contextlib import redirect_stdout

with open('model_evaluation_report.txt', 'w') as f:
    with redirect_stdout(f):
        generate_comprehensive_report(model3, X_train, y_train, X_valid, y_valid, categories, history)

print("Comprehensive evaluation report saved to model_evaluation_report.txt")


In [ ]:
# Add medical imaging evaluation metrics calculation
from sklearn.metrics import cohen_kappa_score, matthews_corrcoef, balanced_accuracy_score

# Calculate Cohen's kappa coefficient (a measure of classification consistency, commonly used in medical research)
kappa = cohen_kappa_score(y_valid, y_pred_bool)

# Calculate Matthews Correlation Coefficient (MCC) - particularly suitable for binary classification evaluation of imbalanced datasets
mcc = matthews_corrcoef(y_valid, y_pred_bool)

# Calculate balanced accuracy (accuracy that takes class imbalance into account)
balanced_acc = balanced_accuracy_score(y_valid, y_pred_bool)

# Calculate sensitivity and specificity for each class
def calculate_sensitivity_specificity(y_true, y_pred, class_idx):
    """Calculate sensitivity and specificity for the specified class"""
    # Convert multiclass problem to binary
    y_true_bin = (y_true == class_idx).astype(int)
    y_pred_bin = (y_pred == class_idx).astype(int)
    
    # Calculate TP, TN, FP, FN
    TP = np.sum((y_true_bin == 1) & (y_pred_bin == 1))
    TN = np.sum((y_true_bin == 0) & (y_pred_bin == 0))
    FP = np.sum((y_true_bin == 0) & (y_pred_bin == 1))
    FN = np.sum((y_true_bin == 1) & (y_pred_bin == 0))
    
    # Calculate sensitivity and specificity
    sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0
    
    return sensitivity, specificity

# Calculate sensitivity and specificity for each class
sensitivities = []
specificities = []

print("\n=== Sensitivity and Specificity ===")
for i in range(len(categories)):
    sens, spec = calculate_sensitivity_specificity(y_valid, y_pred_bool, i)
    sensitivities.append(sens)
    specificities.append(spec)
    print(f"{categories[i]}:")
    print(f"  Sensitivity (Recall): {sens:.4f}")
    print(f"  Specificity: {spec:.4f}")

# Calculate positive predictive value (PPV) and negative predictive value (NPV)
def calculate_ppv_npv(y_true, y_pred, class_idx):
    """Calculate PPV and NPV for the specified class"""
    # Convert multiclass problem to binary
    y_true_bin = (y_true == class_idx).astype(int)
    y_pred_bin = (y_pred == class_idx).astype(int)
    
    # Calculate TP, TN, FP, FN
    TP = np.sum((y_true_bin == 1) & (y_pred_bin == 1))
    TN = np.sum((y_true_bin == 0) & (y_pred_bin == 0))
    FP = np.sum((y_true_bin == 0) & (y_pred_bin == 1))
    FN = np.sum((y_true_bin == 1) & (y_pred_bin == 0))
    
    # Calculate PPV and NPV
    ppv = TP / (TP + FP) if (TP + FP) > 0 else 0  # Also known as precision
    npv = TN / (TN + FN) if (TN + FN) > 0 else 0
    
    return ppv, npv

# Calculate PPV and NPV for each class
ppvs = []
npvs = []

print("\n=== Predictive Values ===")
for i in range(len(categories)):
    ppv, npv = calculate_ppv_npv(y_valid, y_pred_bool, i)
    ppvs.append(ppv)
    npvs.append(npv)
    print(f"{categories[i]}:")
    print(f"  Positive Predictive Value (PPV/Precision): {ppv:.4f}")
    print(f"  Negative Predictive Value (NPV): {npv:.4f}")

# Print global evaluation metrics
print("\n=== Global Evaluation Metrics ===")
print(f"Cohen's Kappa: {kappa:.4f}")
print(f"Matthews Correlation Coefficient: {mcc:.4f}")
print(f"Balanced Accuracy: {balanced_acc:.4f}")

# Calculate diagnostic accuracy metrics
print("\n=== Diagnostic Accuracy Metrics ===")
print(f"Macro-average Sensitivity: {np.mean(sensitivities):.4f}")
print(f"Macro-average Specificity: {np.mean(specificities):.4f}")
print(f"Macro-average Positive Predictive Value: {np.mean(ppvs):.4f}")
print(f"Macro-average Negative Predictive Value: {np.mean(npvs):.4f}")


In [ ]:
# Medical imaging specific visualization methods
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 1. Draw radar chart showing model performance on various metrics
def plot_radar_metrics(categories, metrics_dict):
    """
    Draw radar chart showing multiple metrics for each class
    
    Parameters:
        categories: List of class names
        metrics_dict: Dictionary of metrics with arrays/lists of values
    """
    # Set up the figure
    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111, polar=True)
    
    # Print dimensions for debugging
    print(f"Categories: {categories}, length: {len(categories)}")
    for metric, values in metrics_dict.items():
        print(f"Metric '{metric}' values: {values}, length: {len(values)}")
    
    # Ensure all metrics have the same dimensions as categories
    for metric in list(metrics_dict.keys()):
        if len(metrics_dict[metric]) != len(categories):
            print(f"WARNING: Metric '{metric}' has {len(metrics_dict[metric])} values but there are {len(categories)} categories.")
            print(f"Adjusting '{metric}' to match category length.")
            
            # Option 1: Truncate if too long
            if len(metrics_dict[metric]) > len(categories):
                metrics_dict[metric] = metrics_dict[metric][:len(categories)]
            
            # Option 2: Pad with zeros if too short
            elif len(metrics_dict[metric]) < len(categories):
                padding = np.zeros(len(categories) - len(metrics_dict[metric]))
                metrics_dict[metric] = np.concatenate([metrics_dict[metric], padding])
    
    # Get angles
    angles = np.linspace(0, 2*np.pi, len(categories), endpoint=False).tolist()
    angles += angles[:1]  # Close the plot
    
    # Draw each metric
    metrics = list(metrics_dict.keys())
    colors = ['b', 'r', 'g', 'purple']
    
    for i, metric in enumerate(metrics):
        values = metrics_dict[metric].copy()  # Create a copy to avoid modifying the original
        values = np.append(values, values[0])  # Close the plot (correct way)
        
        # Double check dimensions
        if len(angles) != len(values):
            print(f"ERROR: angles has {len(angles)} elements but values has {len(values)} elements")
            continue
            
        ax.plot(angles, values, 'o-', linewidth=2, color=colors[i % len(colors)], label=metric)
        ax.fill(angles, values, alpha=0.1, color=colors[i % len(colors)])
    
    # Set angle labels
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories)
    
    # Set y-axis range
    ax.set_ylim(0, 1)
    
    # Add legend and title
    plt.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))
    plt.title('Performance Metrics Radar Chart by Class', size=15, y=1.1)
    plt.tight_layout()
    plt.show()

# Prepare radar chart data - explicitly ensure all arrays have the same length
categories_array = np.array(categories)
n_categories = len(categories_array)

# Check dimensions of all metrics
print(f"Categories shape: {categories_array.shape}")
print(f"Sensitivities shape: {np.array(sensitivities).shape}")
print(f"Specificities shape: {np.array(specificities).shape}")
print(f"PPVs shape: {np.array(ppvs).shape}")
print(f"F1 scores shape: {np.array(class_f1).shape}")

# Ensure all metrics have the correct dimensions
sensitivities_array = np.array(sensitivities)
specificities_array = np.array(specificities)
ppvs_array = np.array(ppvs)
f1_array = np.array(class_f1)

# Truncate or pad if needed
if len(sensitivities_array) != n_categories:
    sensitivities_array = np.resize(sensitivities_array, n_categories)
if len(specificities_array) != n_categories:
    specificities_array = np.resize(specificities_array, n_categories)
if len(ppvs_array) != n_categories:
    ppvs_array = np.resize(ppvs_array, n_categories)
if len(f1_array) != n_categories:
    f1_array = np.resize(f1_array, n_categories)

# Create metrics dictionary with corrected arrays
radar_metrics = {
    'Sensitivity': sensitivities_array,
    'Specificity': specificities_array,
    'PPV': ppvs_array,
    'F1 Score': f1_array
}

# Call the function with corrected data
plot_radar_metrics(categories, radar_metrics)

# 2. Draw diagnostic performance comparison chart - fixed to ensure arrays have compatible shapes
plt.figure(figsize=(14, 7))
x = np.arange(len(categories))
width = 0.2

# Ensure all arrays have the same length as x
sens_array = sensitivities_array  # Using the corrected array from above
spec_array = specificities_array
ppv_array = ppvs_array
npv_array = np.array(npvs)

# Resize npv_array if needed
if len(npv_array) != len(x):
    npv_array = np.resize(npv_array, len(x))

# Draw bar chart
plt.bar(x - width*1.5, sens_array, width, label='Sensitivity', color='skyblue')
plt.bar(x - width/2, spec_array, width, label='Specificity', color='salmon')
plt.bar(x + width/2, ppv_array, width, label='PPV', color='lightgreen')
plt.bar(x + width*1.5, npv_array, width, label='NPV', color='purple')

# Set chart properties
plt.xlabel('Class', fontsize=12)
plt.ylabel('Metric Value', fontsize=12)
plt.title('Diagnostic Performance Metrics by Class', fontsize=14)
plt.xticks(x, categories)
plt.ylim(0, 1.1)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


In [ ]:
# Clinical utility analysis with fixed cohen_kappa_score implementation
def clinical_utility_analysis(y_true, y_pred, y_pred_proba, categories):
    """Calculate clinical utility related metrics"""
    # Use a valid weights parameter for cohen_kappa_score
    # Valid options are: None, 'linear', 'quadratic'
    # Instead of custom weights matrix, use the standard quadratic weighting
    weighted_kappa = cohen_kappa_score(y_true, y_pred, weights='quadratic')
    
    # For each malignant case, calculate model's confidence
    malignant_indices = np.where(y_true == 1)[0]
    malignant_confidences = y_pred_proba[malignant_indices, 1] if len(malignant_indices) > 0 else np.array([])
    avg_malignant_confidence = np.mean(malignant_confidences) if len(malignant_confidences) > 0 else 0
    
    # Malignant case detection rate
    malignant_detection_rate = np.sum((y_true == 1) & (y_pred == 1)) / np.sum(y_true == 1) if np.sum(y_true == 1) > 0 else 0
    
    print("\n=== Clinical Utility Analysis ===")
    print(f"Weighted Cohen's Kappa (quadratic weights): {weighted_kappa:.4f}")
    print(f"Average confidence for malignant cases: {avg_malignant_confidence:.4f}")
    print(f"Malignant case detection rate: {malignant_detection_rate:.4f}")
    
    # Calculate custom severity-weighted misclassification metric
    # This addresses our need to penalize missed malignancies more heavily
    # without using a custom weights matrix
    severity_matrix = np.ones((3, 3))
    severity_matrix[1, 0] = 2.0  # Malignant misdiagnosed as benign
    severity_matrix[1, 2] = 2.0  # Malignant misdiagnosed as normal
    
    conf_mat = confusion_matrix(y_true, y_pred)
    total_samples = np.sum(conf_mat)
    weighted_errors = 0
    
    for i in range(3):
        for j in range(3):
            if i != j:  # Only count misclassifications
                weighted_errors += conf_mat[i, j] * severity_matrix[i, j]
    
    severity_weighted_error = weighted_errors / total_samples
    print(f"Severity-weighted error rate: {severity_weighted_error:.4f}")
    
    # Missed diagnosis analysis (malignant misdiagnosed as benign or normal)
    missed_diagnoses = np.where((y_true == 1) & (y_pred != 1))[0]
    print(f"\nNumber of missed malignant diagnoses: {len(missed_diagnoses)} / {np.sum(y_true == 1)}")
    
    if len(missed_diagnoses) > 0:
        misdiagnosed_as = {}
        for idx in missed_diagnoses:
            pred = y_pred[idx]
            if categories[pred] in misdiagnosed_as:
                misdiagnosed_as[categories[pred]] += 1
            else:
                misdiagnosed_as[categories[pred]] = 1
        
        print("Malignant cases misdiagnosed as:")
        for category, count in misdiagnosed_as.items():
            print(f"  - {category}: {count}")
    
    # Calculate and render risk-based decision thresholds
    print("\n=== Recommended Decision Thresholds ===")
    # Set different clinical scenarios (e.g., screening, diagnostic confirmation)
    scenarios = {
        "Low-Risk Screening": 0.3,   # Lower cost of missed diagnosis, suitable for initial screening
        "Routine Diagnosis": 0.5,    # Balanced consideration
        "High-Risk Confirmation": 0.7 # Avoid missed diagnoses, accept some false alarms
    }
    
    for scenario, threshold in scenarios.items():
        # Predict using threshold
        pred_with_threshold = np.zeros_like(y_pred)
        for i in range(len(y_pred_proba)):
            max_prob = np.max(y_pred_proba[i])
            if max_prob >= threshold:
                pred_with_threshold[i] = np.argmax(y_pred_proba[i])
            else:
                # When probability below threshold, recommend further examination
                pred_with_threshold[i] = -1
                
        # Calculate valid prediction rate
        valid_predictions = np.sum(pred_with_threshold != -1)
        valid_rate = valid_predictions / len(y_pred)
        
        # Calculate accuracy among valid predictions
        valid_indices = np.where(pred_with_threshold != -1)[0]
        if len(valid_indices) > 0:
            valid_accuracy = np.sum(y_true[valid_indices] == pred_with_threshold[valid_indices]) / len(valid_indices)
        else:
            valid_accuracy = 0
            
        print(f"{scenario} (threshold={threshold}):")
        print(f"  Valid prediction rate: {valid_rate:.4f}")
        print(f"  Accuracy of valid predictions: {valid_accuracy:.4f}")
        print(f"  Proportion of samples recommended for further examination: {1 - valid_rate:.4f}")

# Call clinical utility analysis with fixed implementation
clinical_utility_analysis(y_valid, y_pred_bool, y_pred, categories)


In [ ]:
# Stratified k-fold cross-validation performance assessment
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def cross_validate_model_performance(model, X, y, n_splits=5):
    """Evaluate model performance using stratified k-fold cross-validation"""
    # Initialize stratified k-fold
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    # Metrics storage
    accuracies = []
    precisions = []
    recalls = []
    f1_scores = []
    
    print(f"\n=== {n_splits}-Fold Cross-Validation Performance ===")
    
    # Perform cross-validation
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        print(f"\nFold {fold+1}/{n_splits}")
        
        # Split data
        X_train_fold, X_test_fold = X[train_idx], X[test_idx]
        y_train_fold, y_test_fold = y[train_idx], y[test_idx]
        
        # Train model on this fold
        model.fit(X_train_fold, y_train_fold, epochs=3,batch_size=16, verbose=0)
        
        # Evaluate on test fold
        y_pred_fold = model.predict(X_test_fold)
        y_pred_classes = np.argmax(y_pred_fold, axis=1)
        
        # Calculate metrics
        acc = accuracy_score(y_test_fold, y_pred_classes)
        prec = precision_score(y_test_fold, y_pred_classes, average='macro')
        rec = recall_score(y_test_fold, y_pred_classes, average='macro')
        f1 = f1_score(y_test_fold, y_pred_classes, average='macro')
        
        # Store metrics
        accuracies.append(acc)
        precisions.append(prec)
        recalls.append(rec)
        f1_scores.append(f1)
        
        # Print fold results
        print(f"  Accuracy: {acc:.4f}")
        print(f"  Precision: {prec:.4f}")
        print(f"  Recall: {rec:.4f}")
        print(f"  F1 Score: {f1:.4f}")
    
    # Calculate and print average performance
    print("\nAverage performance across all folds:")
    print(f"  Accuracy: {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}")
    print(f"  Precision: {np.mean(precisions):.4f} ± {np.std(precisions):.4f}")
    print(f"  Recall: {np.mean(recalls):.4f} ± {np.std(recalls):.4f}")
    print(f"  F1 Score: {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")
    
    # Plot performance distribution
    metrics = {
        'Accuracy': accuracies,
        'Precision': precisions,
        'Recall': recalls,
        'F1 Score': f1_scores
    }
    
    plt.figure(figsize=(10, 6))
    
    # Create boxplot
    plt.boxplot([metrics[m] for m in metrics.keys()], labels=metrics.keys())
    plt.title('Performance Distribution Across Folds')
    plt.ylabel('Score')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    
    # Add individual points
    for i, m in enumerate(metrics.keys()):
        x = np.random.normal(i+1, 0.04, size=len(metrics[m]))
        plt.plot(x, metrics[m], 'r.', alpha=0.4)
    
    plt.tight_layout()
    plt.show()
    
    return metrics

# Uncomment to run cross-validation (note: this can be time-consuming)
cross_val_metrics = cross_validate_model_performance(model3, X_valid, y_valid, n_splits=3)


In [ ]:
# Feature visualization and clustering analysis
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans

def visualize_features(model, X, y, categories, layer_name=-2):
    """Extract and visualize features from the specified layer"""
    # Create a model that outputs the features from the specified layer
    feature_model = tf.keras.models.Model(
        inputs=model.input,
        outputs=model.layers[layer_name].output
    )
    
    # Extract features
    features = feature_model.predict(X)
    
    # Apply dimensionality reduction for visualization
    # First PCA to reduce computation time
    if features.shape[1] > 50:
        pca = PCA(n_components=50)
        features_pca = pca.fit_transform(features)
        print(f"PCA explained variance ratio: {sum(pca.explained_variance_ratio_):.4f}")
    else:
        features_pca = features
    
    # Then t-SNE for visualization
    tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(X)//10))
    features_tsne = tsne.fit_transform(features_pca)
    
    # Plot t-SNE visualization
    plt.figure(figsize=(12, 10))
    
    # Plot by true class
    plt.subplot(2, 2, 1)
    for i, category in enumerate(categories):
        indices = np.where(y == i)[0]
        plt.scatter(features_tsne[indices, 0], features_tsne[indices, 1], label=category, alpha=0.6)
    
    plt.title('Feature Space Visualization (t-SNE)')
    plt.xlabel('t-SNE Dimension 1')
    plt.ylabel('t-SNE Dimension 2')
    plt.legend()
    
    # Plot by prediction correctness
    plt.subplot(2, 2, 2)
    y_pred = np.argmax(model.predict(X), axis=1)
    correct = (y_pred == y)
    plt.scatter(features_tsne[correct, 0], features_tsne[correct, 1], 
                label='Correct Predictions', color='green', alpha=0.6)
    plt.scatter(features_tsne[~correct, 0], features_tsne[~correct, 1], 
                label='Incorrect Predictions', color='red', alpha=0.6)
    
    plt.title('Prediction Correctness Visualization')
    plt.xlabel('t-SNE Dimension 1')
    plt.ylabel('t-SNE Dimension 2')
    plt.legend()
    
    # K-means clustering
    n_clusters = len(categories)
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(features_pca)
    
    # Plot by cluster
    plt.subplot(2, 2, 3)
    for i in range(n_clusters):
        indices = np.where(cluster_labels == i)[0]
        plt.scatter(features_tsne[indices, 0], features_tsne[indices, 1], 
                   label=f'Cluster {i+1}', alpha=0.6)
    
    plt.title('Feature Clustering (K-means)')
    plt.xlabel('t-SNE Dimension 1')
    plt.ylabel('t-SNE Dimension 2')
    plt.legend()
    
    # Calculate cluster purity
    purity_table = np.zeros((n_clusters, len(categories)))
    for i in range(n_clusters):
        cluster_indices = np.where(cluster_labels == i)[0]
        for j in range(len(categories)):
            purity_table[i, j] = np.sum(y[cluster_indices] == j) / len(cluster_indices) if len(cluster_indices) > 0 else 0
    
    # Plot cluster purity as a heatmap
    plt.subplot(2, 2, 4)
    sns.heatmap(purity_table, annot=True, fmt='.2f', cmap='Blues',
               xticklabels=categories, yticklabels=[f'Cluster {i+1}' for i in range(n_clusters)])
    plt.title('Cluster Purity (Class Distribution)')
    plt.ylabel('Cluster')
    plt.xlabel('True Class')
    
    plt.tight_layout()
    plt.show()
    
    # Return the features and their reduced representations
    return features, features_pca, features_tsne, cluster_labels

# Extract and visualize features (uncomment to run)
features_data = visualize_features(model3, X_valid, y_valid, categories)


In [ ]:
# Analysis of sample difficulty for the model
def analyze_sample_difficulty(y_true, y_pred_proba, categories):
    """Analyze classification difficulty of samples by the model"""
    # Calculate prediction confidence for each sample
    confidences = np.max(y_pred_proba, axis=1)
    predicted_classes = np.argmax(y_pred_proba, axis=1)
    correct_predictions = (predicted_classes == y_true)
    
    # Create result dataframe
    import pandas as pd
    df = pd.DataFrame({
        'True_Class': [categories[y] for y in y_true],
        'Predicted_Class': [categories[y] for y in predicted_classes],
        'Confidence': confidences,
        'Correct': correct_predictions
    })
    
    # Sort by confidence
    df_sorted = df.sort_values('Confidence')
    
    # Analyze accuracy at different confidence intervals
    confidence_bins = [0, 0.25, 0.5, 0.75, 0.9, 1.0]
    bin_labels = ['0-25%', '25-50%', '50-75%', '75-90%', '90-100%']
    
    df['Confidence_Bin'] = pd.cut(df['Confidence'], confidence_bins, labels=bin_labels)
    bin_accuracy = df.groupby('Confidence_Bin')['Correct'].mean()
    bin_counts = df.groupby('Confidence_Bin')['Correct'].count()
    
    # Plot relationship between confidence and accuracy
    plt.figure(figsize=(14, 6))
    
    # Accuracy bar chart
    plt.subplot(1, 2, 1)
    bin_accuracy.plot(kind='bar', color='skyblue')
    plt.title('Accuracy at Different Confidence Intervals')
    plt.xlabel('Confidence Interval')
    plt.ylabel('Accuracy')
    plt.ylim(0, 1.1)
    
    # Sample count bar chart
    plt.subplot(1, 2, 2)
    bin_counts.plot(kind='bar', color='salmon')
    plt.title('Sample Count at Different Confidence Intervals')
    plt.xlabel('Confidence Interval')
    plt.ylabel('Sample Count')
    
    plt.tight_layout()
    plt.show()
    
    # Return difficult samples (low confidence but correct prediction, or high confidence but incorrect prediction)
    hard_correct = df[(df['Confidence'] < 0.6) & df['Correct']].sort_values('Confidence')
    hard_incorrect = df[(df['Confidence'] > 0.8) & ~df['Correct']].sort_values('Confidence', ascending=False)
    
    print("\n=== Difficult but Correct Samples (Low Confidence) ===")
    if len(hard_correct) > 0:
        print(hard_correct.head(5))
    else:
        print("No such samples")
        
    print("\n=== Incorrectly Classified with High Confidence (Model Confidently Wrong) ===")
    if len(hard_incorrect) > 0:
        print(hard_incorrect.head(5))
    else:
        print("No such samples")
    
    return df_sorted

# Analyze sample difficulty
difficulty_analysis = analyze_sample_difficulty(y_valid, y_pred, categories)
